In [0]:
dbutils.widgets.text("catalog_name", "cjc", "Catalog Name")
dbutils.widgets.text("schema_name", "geo", "Schema Name")
dbutils.widgets.text("volume_name", "data", "Volume Name")
dbutils.widgets.text("config_name", "config", "Config path")

In [0]:
%sql
USE CATALOG ${catalog_name};
--CREATE CATALOG IF NOT EXISTS ${catalog_name};
CREATE SCHEMA IF NOT EXISTS ${schema_name};
USE SCHEMA ${schema_name};
CREATE VOLUME IF NOT EXISTS ${volume_name};
CREATE VOLUME IF NOT EXISTS ${config_name};

# Sedona setup
- [requirements from sedona ](https://sedona.apache.org/latest/setup/databricks/?h=databric#set-up-cluster-config )

In [0]:
catalog_name = dbutils.widgets.get("catalog_name")
schema_name = dbutils.widgets.get("schema_name")
config_name = dbutils.widgets.get("config_name")

# requirements from sedona https://sedona.apache.org/latest/setup/databricks/?h=databric#set-up-cluster-config 

requirements = """
apache-sedona==1.7.0
geopandas==0.11.1
keplergl==0.3.2
pydeck==0.8.0
"""

file_path = f"/Volumes/{catalog_name}/{schema_name}/{config_name}/requirements.txt"

with open(file_path, "w") as file:
    file.write(requirements)

In [0]:
%sh
# Create JAR directory for Sedona
mkdir -p /Workspace/Shared/sedona/1.7.0

# Download the dependencies from Maven into DBFS
curl -o /Workspace/Shared/sedona/1.7.0/geotools-wrapper-1.7.0-28.5.jar "https://repo1.maven.org/maven2/org/datasyslab/geotools-wrapper/1.7.0-28.5/geotools-wrapper-1.7.0-28.5.jar"

curl -o /Workspace/Shared/sedona/1.7.0/sedona-spark-shaded-3.4_2.12-1.7.0.jar "https://repo1.maven.org/maven2/org/apache/sedona/sedona-spark-shaded-3.4_2.12/1.7.0/sedona-spark-shaded-3.4_2.12-1.7.0.jar"

In [0]:
%sh 
# Create init script
cat > /Volumes/cjc/geo/config/sedona-init.sh <<'EOF'
#!/bin/bash
#
# File: sedona-init.sh
#
# On cluster startup, this script will copy the Sedona jars to the cluster's default jar directory.

cp /Workspace/Shared/sedona/1.7.0/*.jar /databricks/jars
pip install -r /Volumes/cjc/geo/config/requirements.txt

EOF